# Aprendizado de Máquina — Aula prática E3

## Processamento de Linguagem Natural e Classificação

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Última aula prática do curso, e ela não traz método novo. Traz um **tipo de dado**
novo — texto — e mostra que tudo o que construímos nas doze aulas anteriores se
aplica sem alteração, desde que se resolva primeiro um problema:

> **um algoritmo de aprendizado recebe uma matriz de números. Uma mensagem de
> texto não é uma matriz de números. Alguém precisa decidir como transformá-la — e
> essa decisão é o modelo tanto quanto o classificador.**

O problema é filtrar *spam*: 5.572 mensagens de SMS rotuladas. A matriz que vamos
construir tem milhares de colunas e é 99,9% zeros, o que faz deste o exemplo mais
puro da maldição da dimensionalidade da Aula 05 — e, ao mesmo tempo, o exemplo em
que ela **não** atrapalha, pelo motivo que a própria Aula 05 antecipou.

### Objetivos

Ao final deste notebook você deve ser capaz de:

- ler e limpar um arquivo de texto real, com colunas-lixo e codificação estranha;
- construir uma matriz documento–termo com `CountVectorizer` e medir a esparsidade;
- explicar o que o TF-IDF faz e por que ele costuma ajudar;
- comparar `MultinomialNB`, logística e SVM linear no mesmo texto;
- ler as palavras mais discriminativas e desconfiar delas;
- montar o `Pipeline` completo com validação cruzada e as métricas da Aula 09.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

Os objetos novos vêm de `sklearn.feature_extraction.text` — são eles que fazem a
ponte entre texto e matriz.

In [ ]:
import sklearn.model_selection as skm
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, classification_report,
                             confusion_matrix, roc_auc_score)
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

In [ ]:
import warnings
warnings.filterwarnings("ignore")

---
## 2. O arquivo, e a limpeza que ele exige

O `spam.csv` é a *SMS Spam Collection*, uma base clássica. Como quase todo arquivo
de texto que circula, ele tem duas complicações de formato.

In [ ]:
import os

_nome = "spam.csv"
_local = os.path.join("..", "..", "recursos", "dados", _nome)   # repositorio clonado
_url = ("https://raw.githubusercontent.com/HugoCarvalhoUFRJ/ap-maq/"
        "refs/heads/refactoring-baby/recursos/dados/") + _nome  # fallback (ex.: Colab)
_fonte = _local if os.path.exists(_local) else _url

bruto = pd.read_csv(_fonte, encoding="latin-1")
print("colunas:", list(bruto.columns))
print(f"dimensoes: {bruto.shape}")
print(f"\nnao nulos por coluna:\n{bruto.notna().sum().to_string()}")

As duas complicações. A primeira é a **codificação**: sem `encoding="latin-1"` a
leitura falha, porque o arquivo não está em UTF-8 — é o tipo de detalhe que consome
uma tarde de quem não sabe procurar.

A segunda são as três colunas `Unnamed`, quase inteiramente vazias: resquício de
mensagens que continham vírgulas e por isso "vazaram" para colunas extras. Ficamos
com as duas primeiras.

In [ ]:
dados = bruto.iloc[:, :2].copy()
dados.columns = ["rotulo", "texto"]
dados["y"] = (dados["rotulo"] == "spam").astype(int)

print(f"{len(dados)} mensagens")
print(dados["rotulo"].value_counts().to_string())
print(f"prevalencia de spam: {dados['y'].mean():.4f}")
print(f"\ncomprimento medio (caracteres): ham {dados.loc[dados.y==0,'texto'].str.len().mean():.0f}, "
      f"spam {dados.loc[dados.y==1,'texto'].str.len().mean():.0f}")
dados.head(4)

Já aqui há um sinal: mensagens de *spam* são bem mais longas. Guarde — é uma
covariável de graça que o saco de palavras não captura diretamente.

---
## 3. Do texto à matriz: o saco de palavras

A representação mais simples que existe: cada **palavra distinta** do corpus vira
uma coluna, e cada mensagem vira uma linha com a contagem de cada palavra. É o
*bag of words* — "saco" porque a ordem das palavras é jogada fora.

*"não é spam"* e *"é spam, não"* viram exatamente a mesma linha. Perde-se muito. E
funciona surpreendentemente bem.

In [ ]:
vec = CountVectorizer()
M = vec.fit_transform(dados["texto"])
y = dados["y"].to_numpy()

densidade = M.nnz / (M.shape[0] * M.shape[1])
print(f"matriz documento-termo: {M.shape[0]} mensagens x {M.shape[1]} palavras")
print(f"celulas: {M.shape[0] * M.shape[1]:,}")
print(f"celulas nao nulas: {M.nnz:,}   ->  densidade = {densidade:.5%}")
print(f"palavras distintas por mensagem, em media: {M.nnz / M.shape[0]:.1f}")
print(f"\ntipo do objeto: {type(M).__name__}  (esparso, nao um array denso)")
print(f"memoria como esparsa: {(M.data.nbytes + M.indices.nbytes + M.indptr.nbytes)/1e6:.1f} MB")
print(f"memoria se fosse densa: {M.shape[0] * M.shape[1] * 8 / 1e6:.0f} MB")

Aqui está a Aula 05 em estado puro: **8.700 colunas para 5.572 observações**, com
$d > n$, e 99,9% da matriz é zero. Se armazenássemos densamente seriam centenas de
megabytes para guardar quase nada — daí a matriz esparsa, que só guarda as posições
não nulas.

E é aqui que a Aula 05 também dá a resposta de por que isso funciona: o problema é
**esparso**. Quase nenhuma palavra importa, e os métodos que fazem seleção de
variáveis — ou que, como a logística com $\ell_2$, encolhem coeficientes inúteis —
operam na dimensão efetiva. O KNN, que somaria as 8.700 coordenadas com peso igual,
seria um desastre. Vamos confirmar isso na Seção 6.

In [ ]:
sub = M[:180, :420].toarray() > 0
fig, (ax1, ax2) = subplots(1, 2, figsize=(9.5, 3.4),
                           gridspec_kw={"width_ratios": [1, 1.1]})
ax1.imshow(sub, cmap="Greys", aspect="auto", interpolation="nearest")
ax1.set_xlabel(f"palavras (as 420 primeiras de {M.shape[1]})")
ax1.set_ylabel("mensagens (as 180 primeiras)")
ax1.set_title(f"cada ponto e' uma palavra presente\ndensidade global {densidade:.3%}",
              fontsize=9)
ax1.grid(False)

freq = np.asarray(M.sum(axis=0)).ravel()
ordem = np.argsort(freq)[::-1]
ax2.loglog(np.arange(1, len(freq) + 1), freq[ordem])
ax2.set_xlabel("posicao no ranque de frequencia")
ax2.set_ylabel("total de ocorrencias")
ax2.set_title("lei de Zipf: reta em log-log", fontsize=9)

nomes = np.array(vec.get_feature_names_out())
print("as 10 palavras mais frequentes:", list(nomes[ordem[:10]]))
print(f"palavras que aparecem UMA unica vez: {(freq == 1).sum()} "
      f"({(freq == 1).mean():.1%} do vocabulario)")

Duas leituras. A da esquerda é a esparsidade, visível. A da direita é a **lei de
Zipf**: a frequência de uma palavra é aproximadamente inversamente proporcional à
sua posição no ranque, o que em escala log–log é uma reta.

A consequência prática de Zipf é a que interessa: metade do vocabulário aparece uma
única vez no corpus inteiro. Essas colunas não podem ensinar nada — não há como
estimar coisa alguma a partir de uma observação — e só engordam a matriz. É para
elas que serve o `min_df`.

---
## 4. TF-IDF, e o que ele conserta

A contagem crua tem um problema óbvio: a palavra *"the"* aparece em quase toda
mensagem e não distingue nada, enquanto *"claim"* aparece em poucas e distingue
muito. O **TF-IDF** corrige isso multiplicando a contagem pelo inverso da frequência
nos documentos:

$$\text{tf-idf}(t, d) = \underbrace{\text{contagem}(t, d)}_{\text{tf}}
   \times \underbrace{\log\frac{1+n}{1+\text{df}(t)} + 1}_{\text{idf}},$$

onde $\text{df}(t)$ é em quantos documentos o termo aparece. Palavras onipresentes
recebem peso próximo de zero.

In [ ]:
tfidf = TfidfVectorizer()
T = tfidf.fit_transform(dados["texto"])
idf = tfidf.idf_
nomes_t = np.array(tfidf.get_feature_names_out())

ordem_idf = np.argsort(idf)
print("as 8 palavras de MENOR idf (aparecem em quase tudo, pesam pouco):")
print("   ", list(nomes_t[ordem_idf[:8]]))
print("as 8 de MAIOR idf (raras, pesam muito):")
print("   ", list(nomes_t[ordem_idf[-8:]]))
print(f"\nfaixa do idf: de {idf.min():.2f} a {idf.max():.2f}")

Vale também olhar o efeito do `min_df`, que descarta termos raros demais. Ele reduz
drasticamente o vocabulário e — nesta base — não custa desempenho, porque o que ele
joga fora é justamente a cauda de Zipf.

In [ ]:
for m in [1, 2, 3, 5, 10]:
    v = CountVectorizer(min_df=m).fit(dados["texto"])
    print(f"min_df = {m:2d}: {len(v.vocabulary_):5d} palavras "
          f"({len(v.vocabulary_)/M.shape[1]:.1%} do vocabulario original)")

---
## 5. Três classificadores no mesmo texto

Agora o de sempre: separar treino e teste, montar `Pipeline` e comparar. Repare que
o vetorizador é uma **etapa do pipeline** — ele aprende o vocabulário e os pesos IDF
dos dados, então cabe exatamente na regra da Aula 07.

In [ ]:
texto_tr, texto_te, y_tr, y_te = skm.train_test_split(
    dados["texto"], y, test_size=0.3, random_state=0, stratify=y)

candidatos = {
    "Bayes ingenuo multinomial": MultinomialNB(),
    "regressao logistica": LogisticRegression(max_iter=5000),
    "SVM linear": LinearSVC(C=1),
}

linhas = {}
for nome, clf in candidatos.items():
    pipe = Pipeline([("tfidf", TfidfVectorizer(min_df=2)), ("clf", clf)]).fit(texto_tr, y_tr)
    escore = (pipe.predict_proba(texto_te)[:, 1] if hasattr(pipe, "predict_proba")
              else pipe.decision_function(texto_te))
    pred = pipe.predict(texto_te)
    linhas[nome] = {"acuracia": (pred == y_te).mean(),
                    "AUC": roc_auc_score(y_te, escore),
                    "AP": average_precision_score(y_te, escore),
                    "revocacao (spam)": (pred[y_te == 1] == 1).mean(),
                    "precisao (spam)": (y_te[pred == 1] == 1).mean()}
pd.DataFrame(linhas).T.round(4)

Os três vão bem, e a comparação interessante não está na acurácia — com 13% de
*spam*, a Aula 09 já nos ensinou a desconfiar dela. Está na dupla
precisão/revocação, e o motivo é o custo assimétrico: **deixar passar um spam é um
aborrecimento; jogar na lixeira uma mensagem legítima é um desastre.** Você quer
precisão altíssima na classe *spam*, mesmo à custa de revocação.

Vale notar por que o Bayes ingênuo funciona aqui apesar de a suposição dele ser
obviamente falsa — palavras de uma mensagem não são independentes. A resposta é a
da Aula 09: ele **ordena** bem mesmo estimando probabilidades ruins, e para
classificar só a ordem importa. Foi por isso que ele dominou os filtros de spam nos
anos 2000.

In [ ]:
melhor = Pipeline([("tfidf", TfidfVectorizer(min_df=2)),
                   ("clf", LogisticRegression(max_iter=5000, C=10))]).fit(texto_tr, y_tr)
pred = melhor.predict(texto_te)

print(pd.DataFrame(confusion_matrix(y_te, pred), index=["ham real", "spam real"],
                   columns=["previu ham", "previu spam"]).to_string())
print()
print(classification_report(y_te, pred, target_names=["ham", "spam"], digits=3))

falsos_positivos = np.where((pred == 1) & (y_te == 0))[0]
print(f"mensagens legitimas classificadas como spam: {len(falsos_positivos)}")
for i in falsos_positivos[:3]:
    print(f"   - {texto_te.iloc[i][:90]!r}")

---
## 6. E o KNN? A Aula 05, confirmada

A Seção 3 afirmou que o KNN seria um desastre aqui, e a razão era a esparsidade:
ele soma as 8.700 coordenadas com peso igual, e quase todas são zero para qualquer
par de mensagens. Vamos conferir em vez de acreditar.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

vet = TfidfVectorizer(min_df=2)
Xtr_v = vet.fit_transform(texto_tr)
Xte_v = vet.transform(texto_te)

linhas_knn = []
for k in [1, 5, 25]:
    knn = KNeighborsClassifier(n_neighbors=k).fit(Xtr_v, y_tr)
    pk = knn.predict(Xte_v)
    linhas_knn.append({"metodo": f"KNN (k={k})", "acuracia": (pk == y_te).mean(),
                       "revocacao (spam)": (pk[y_te == 1] == 1).mean()})
linhas_knn.append({"metodo": "logistica", "acuracia": (pred == y_te).mean(),
                   "revocacao (spam)": (pred[y_te == 1] == 1).mean()})
linhas_knn.append({"metodo": "chutar ham sempre",
                   "acuracia": 1 - y_te.mean(), "revocacao (spam)": 0.0})
pd.DataFrame(linhas_knn).set_index("metodo").round(4)

Compare a revocação. É essa coluna que mostra o que a Aula 05 previu: num espaço
com milhares de coordenadas quase todas nulas, "os $k$ vizinhos mais próximos" de
uma mensagem são uma seleção pouco informativa, e o método deixa passar uma fração
grande dos *spams*.

A logística, com a mesma matriz, encontra quase todos. A diferença entre os dois não
é sofisticação — é que um deles **pondera** as colunas e o outro as trata todas
igual.

---
## 7. O que o modelo aprendeu

Um modelo linear sobre saco de palavras tem uma vantagem rara: cada coeficiente
corresponde a uma palavra, e dá para ler.

In [ ]:
vocab = np.array(melhor.named_steps["tfidf"].get_feature_names_out())
coef = melhor.named_steps["clf"].coef_.ravel()
ordem_c = np.argsort(coef)

fig, ax = subplots(figsize=(6.4, 4.0))
top_spam, top_ham = ordem_c[-15:], ordem_c[:15]
pos = np.arange(15)
ax.barh(pos + 0.5, coef[top_spam], color="crimson", label="empurra para spam")
ax.barh(pos - 16.0, coef[top_ham], color="steelblue", label="empurra para ham")
ax.set_yticks(list(pos + 0.5) + list(np.arange(15) - 16.0))
ax.set_yticklabels(list(vocab[top_spam]) + list(vocab[top_ham]), fontsize=8)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("coeficiente da regressao logistica")
ax.legend(fontsize=8, loc="lower right")

As palavras que empurram para *spam* são reconhecíveis: prêmios, urgência,
números de telefone, "grátis". As que empurram para legítimo são o vocabulário de
conversa cotidiana.

E aqui cabe a última advertência do curso, que é a mesma da Aula 06, Seção 10.
**Esses coeficientes descrevem o modelo, não o mundo.** Eles refletem este corpus,
desta época, deste idioma. Uma palavra pode ter coeficiente alto por aparecer em
três mensagens de spam por acaso; palavras correlacionadas dividem o crédito de
maneira arbitrária; e nada aqui é causal. Use a lista para entender o que o modelo
está fazendo — e para desconfiar dele —, não como conhecimento sobre spam.

---
## 8. O pipeline completo, com tudo o que o curso ensinou

Para fechar, uma busca que trata as decisões de **representação do texto** como
hiperparâmetros — porque é exatamente o que elas são — junto com as do
classificador, tudo dentro da validação cruzada, com o `scoring` escolhido pela
pergunta que estamos respondendo.

In [ ]:
pipe = Pipeline([("tfidf", TfidfVectorizer()), ("clf", LogisticRegression(max_iter=5000))])
grade = {
    "tfidf__min_df": [1, 2, 5],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__sublinear_tf": [False, True],
    "clf__C": [1, 10, 100],
}
busca = skm.GridSearchCV(pipe, grade, cv=5, scoring="average_precision",
                         n_jobs=-1).fit(texto_tr, y_tr)

print("melhores escolhas:")
for chave, valor in busca.best_params_.items():
    print(f"   {chave:24s} = {valor}")
print(f"\nAP de CV no vencedor: {busca.best_score_:.4f}")
print(f"combinacoes avaliadas: {len(busca.cv_results_['mean_test_score'])}")

In [ ]:
p_te = busca.predict_proba(texto_te)[:, 1]
pred_te = busca.predict(texto_te)
print(f"no conjunto de teste, tocado uma unica vez:")
print(f"   acuracia : {(pred_te == y_te).mean():.4f}")
print(f"   AUC      : {roc_auc_score(y_te, p_te):.4f}")
print(f"   AP       : {average_precision_score(y_te, p_te):.4f}")

# o corte que respeita o custo assimetrico: um ham na lixeira custa muito mais
for c_fp, c_fn in [(1, 1), (20, 1), (100, 1)]:
    corte = c_fp / (c_fp + c_fn)
    q = p_te >= corte
    print(f"\ncusto FP={c_fp:3d}, FN={c_fn}: corte otimo = {corte:.3f}")
    print(f"   spams bloqueados: {(q & (y_te == 1)).sum()} de {int(y_te.sum())}"
          f"   hams na lixeira: {(q & (y_te == 0)).sum()}")

A última tabela é o curso inteiro em oito linhas. Há um modelo que estima
$P(\text{spam}\mid \text{texto})$; há uma decisão, que é onde cortar; e há um custo
que ninguém além de quem usa o filtro pode informar. Quando o custo de bloquear uma
mensagem legítima sobe, o corte sobe junto, e trocamos *spams* bloqueados por
caixas de entrada intactas.

Nenhuma dessas três coisas é opcional, e as três são etapas diferentes. Foi o que a
Aula 01 chamou de risco, a Aula 03 de estimar risco sem se enganar, e a Aula 09 de
decidir.

> **Sua vez.** Acrescente à matriz duas colunas que o saco de palavras não captura:
> o **comprimento** da mensagem e a **proporção de dígitos** nela (lembre da Seção 2
> — spams são bem mais longos). Use um `ColumnTransformer` (Aula 07) para juntar as
> colunas de texto com as duas numéricas. O desempenho melhora? Meça por AP, não por
> acurácia.

---
## Resumo

| Conceito | Onde apareceu | O que vimos |
|---|---|---|
| limpeza | §2 | `encoding="latin-1"` e três colunas-lixo de vírgulas vazadas |
| saco de palavras | §3 | 8.700 colunas para 5.572 mensagens, densidade de 0,1% |
| matriz esparsa | §3 | megabytes contra centenas de megabytes |
| lei de Zipf | §3 | metade do vocabulário aparece uma única vez |
| TF-IDF | §4 | pesa a contagem pelo inverso da frequência nos documentos |
| `min_df` | §4 | corta a cauda de Zipf sem custo de desempenho |
| três classificadores | §5 | o Bayes ingênuo funciona por **ordenar** bem, não por acertar probabilidades |
| custo assimétrico | §5, §8 | precisão importa mais que revocação: um ham na lixeira é caro |
| KNN em texto | §6 | a maldição da Aula 05, confirmada — a revocação despenca |
| coeficientes | §7 | legíveis, e mesmo assim descrevem o modelo, não o mundo |
| pipeline completo | §8 | representação do texto é hiperparâmetro, e entra na busca |

**Leitura recomendada.** [AME] §8.1.3 (classificação de texto), §3.8 (o exemplo das
resenhas da Amazon, que é a pergunta com que a Aula 05 abriu) e §8.8. O Apêndice
A.2–A.3 do [AME] cobre a representação de texto com mais cuidado.

**Encerramento.** Este é o fim das aulas práticas. Vale reler, agora, a Seção 8: ela
usa validação cruzada (Aula 03), `Pipeline` sem vazamento (Aula 07), um modelo
linear regularizado (Aula 02), métricas para classe rara (Aula 09) e um corte
derivado de custos (Aula 09) — em um problema com $d > n$ que só é tratável pela
esparsidade (Aula 05). Nada disso é sobre texto. É sobre método, e o texto foi só a
desculpa para usá-lo todo de uma vez.

**Para praticar.** `Lista teorica E3.pdf` (teórica, com gabarito) e
`Lista prática E3.ipynb` (prática, para completar as lacunas), nesta mesma
pasta.